# gold layer------------------|

## step 3 : Operations and Data Analysis

In [0]:
import sys
import pyspark.sql.functions as f


# 1-
try:
    print("Starting Gold Layer Processing...")
    silver_path = "/Volumes/workspace/earthquake_db/silver_zone/earthquakes_delta"
    gold_path = "/Volumes/workspace/earthquake_db/gold_zone/earthquakes_gold"
    checkpoint_path = "/Volumes/workspace/earthquake_db/gold_zone/_checkpoints/earthquakes_gold_stream"

    silver_stream_df = spark.readStream\
        .format("delta")\
        .load(silver_path)

    # 2-   
    gold_transformed_df = silver_stream_df\
        .filter(f.col("earthquake_id").isNotNull())\
        .withColumn("magnitude", f.round(f.col("magnitude" ), 2 ))\
        .withColumn("magnitude_class",
          f.when(f.col("magnitude") < 3.0, "Minor")
           .when((f.col("magnitude") >= 3.0) & (f.col("magnitude") <= 5.0 ), "Moderate" )
           .otherwise("Major")
        )\
        .filter(f.col("event_time").isNotNull())\
        .withColumn("event_time_earthquake",f.to_timestamp(f.from_unixtime(f.col("event_time") / 1000)))\
        .withColumn("is_tsunami", f.when(f.col("is_tsunami") == 1, True).otherwise(False))\
        .withColumn("title" , f.coalesce(f.col("title"), f.lit("No title")))\
        .withColumn("longitude", f.round(f.col("longitude"), 4))\
        .withColumn("latitude", f.round(f.col("latitude"), 4))\
        .withColumn("depth", f.round(f.col("depth"), 2))\
        .withColumn("ingested_at", f.current_timestamp()) \
        .drop("event_time")\
        .select(
        "earthquake_id",
        "magnitude",
        "magnitude_class",        # الترتيب بجانب magnitude
        "event_time_earthquake",
        "is_tsunami",
        "title",
        "longitude",
        "latitude",
        "depth",
        "ingested_at"             # عمود وقت المعالجة في النهاية
    )


    #3-

    query = gold_transformed_df.writeStream\
        .format("delta")\
        .outputMode("append")\
        .trigger(availableNow=True)\
        .option("checkpointLocation", checkpoint_path)\
        .start(gold_path)


    query.awaitTermination()

    print("Gold Layer Processed Successfully!")

    #4_
except Exception as e:
    print(f"CRITICAL ERROR in Gold Layer Pipeline: {str(e)}", file=sys.stderr)
    
    raise e
